# 04. Prediction and Intervals

For a new predictor vector $\mathbf x_0$, the fitted mean response is

$$\widehat{\mu}_{Y|\mathbf x_0}=\mathbf x_0'\widehat{\boldsymbol\beta}.$$

A confidence interval estimates the mean response at that predictor setting. A prediction interval estimates a future individual response at that setting. The prediction interval is wider because it includes both uncertainty in the fitted mean and the noise of a new observation.

By the end of this notebook, you should be able to:

- compute a fitted mean response for a new predictor setting;
- distinguish a confidence interval for the mean response from a prediction interval for a future observation;
- explain how the design-distance value $h_0$ affects interval width.


Let

$$\mathbf x_0=[1,x_{01},x_{02},\ldots,x_{0k}]'$$

and define the design-distance value

$$h_0=\mathbf x_0'(\mathbf X'\mathbf X)^{-1}\mathbf x_0.$$

Then the usual 100$(1-\alpha)$% intervals are

$$\text{CI for mean response: }\quad \widehat y_0 \pm t_{1-\alpha/2,\,n-k-1}\sqrt{MSE\,h_0},$$

$$\text{PI for a future response: }\quad \widehat y_0 \pm t_{1-\alpha/2,\,n-k-1}\sqrt{MSE(1+h_0)}.$$

The extra `1` in the prediction interval is the future-observation error variance.


In [1]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


Running outside JupyterLite; assuming packages are already installed.


In [2]:
ads = pd.read_csv("data/advertising_sales.csv")
model = smf.ols("Sales ~ RadioSpend + PrintSpend", data=ads).fit()

new_campaign = pd.DataFrame({"RadioSpend": [4.5], "PrintSpend": [3.0]})
pred = model.get_prediction(new_campaign).summary_frame(alpha=0.05)
pred

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,23.253242,0.270061,22.683464,23.82302,21.19464,25.311844


In [3]:
mean_hat = pred.loc[0, "mean"]
ci_low, ci_high = pred.loc[0, ["mean_ci_lower", "mean_ci_upper"]]
pi_low, pi_high = pred.loc[0, ["obs_ci_lower", "obs_ci_upper"]]

print(f"Estimated mean sales: {mean_hat:.3f}")
print(f"95% CI for mean sales: ({ci_low:.3f}, {ci_high:.3f})")
print(f"95% prediction interval for one future sales value: ({pi_low:.3f}, {pi_high:.3f})")

Estimated mean sales: 23.253
95% CI for mean sales: (22.683, 23.823)
95% prediction interval for one future sales value: (21.195, 25.312)


In [4]:
from checks import check_interval_order
print(check_interval_order(ci_low, mean_hat, ci_high, label="mean-response confidence interval"))
print(check_interval_order(pi_low, mean_hat, pi_high, label="individual prediction interval"))


Correct: mean-response confidence interval contains the estimate in the expected order.
Correct: individual prediction interval contains the estimate in the expected order.


## Why the Intervals Depend on Location

The term $\mathbf x_0'(\mathbf X'\mathbf X)^{-1}\mathbf x_0$ measures how far the new predictor setting is from the center of the observed design. Predictions near the center of the data are usually more precise than extrapolations.


In [5]:
X = model.model.exog
x0 = np.array([1, new_campaign.loc[0, "RadioSpend"], new_campaign.loc[0, "PrintSpend"]])
h0 = x0 @ np.linalg.inv(X.T @ X) @ x0
print(f"Design distance h0 = {h0:.4f}")

Design distance h0 = 0.0830


Try changing `RadioSpend` and `PrintSpend` in `new_campaign`. Watch how the fitted mean and intervals respond, especially when you move far outside the observed data range.